<div dir="rtl">

# بنچمارک مدل‌های برداری‌سازی واقعی روی متن فارسی

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/snikookar/Persian-RAG-Academy/blob/main/notebooks/benchmark_real_embeddings.ipynb)

این دفترچه (Notebook) همراه [درس ۰۵ — برداری‌سازی (Embeddings)](https://github.com/snikookar/Persian-RAG-Academy/blob/main/lessons/05-embeddings/README.fa.md) است.
چند مدل برداری‌سازی چندزبانه از Hugging Face را روی **همان** پیکره و **همان** پرسش‌های دوره می‌سنجد و کنار روش‌های آفلاین دوره (`HashingEmbedding` و TF-IDF) می‌گذارد.

- پیکره: ۲۵ سند فارسی ساختگی که برای همین دوره نوشته شده‌اند (`data/fixtures/corpus`).
- دو مجموعه پرسش: `qa.jsonl` (پرسش‌هایی با واژه‌های خود اسناد) و `qa_paraphrase.jsonl` (بازنویسی‌شده با مترادف، زبان محاوره و غلط تایپی).
- سنجه‌ها در سطح قطعه (Passage-level): Recall@k، MRR و nDCG@3 از `parsrag.eval`؛ یعنی قطعهٔ بازیابی‌شده باید **خود جملهٔ شاهد** را داشته باشد، نه فقط از سند درست باشد.
- قطعه‌بندی و نرمال‌سازی همان پیش‌فرض دوره است (قطعه‌بندی بازگشتی ۴۰۰ نویسه با سرتیتر، `parsrag.normalize`)، پس **تنها چیزی که عوض می‌شود مدل برداری‌سازی است**.

> **صادقانه:** هیچ عددی برای مدل‌های واقعی در این مخزن از قبل نوشته نشده است. هر عددی که این‌جا می‌بینید را خود دفترچه روی سخت‌افزار شما اندازه گرفته است.
> اگر خواستید نتیجه را با دیگران به اشتراک بگذارید، بخش آخر («نتیجه را برگردانید») را ببینید.

**اجرا:** در Colab از منوی *Runtime → Change runtime type* یک GPU رایگان (مثلاً T4) انتخاب کنید؛ با CPU هم اجرا می‌شود، فقط کندتر (پیکره کوچک است: حدود ۱۰۰ قطعه و ۱۰۵ پرسش).

</div>

<div dir="rtl">

## ۱. نصب `parsrag` از مخزن GitHub

این سلول مخزن را کلون می‌کند (داده‌های نمونه در آن است) و `parsrag` را همراه افزونه‌های `st` (sentence-transformers) و `openai` نصب می‌کند.
اگر دفترچه را روی کامپیوتر خودتان و از داخل مخزن اجرا می‌کنید، کلون انجام نمی‌شود و فقط نصب می‌شود.

</div>

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/snikookar/Persian-RAG-Academy"

local = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "src" / "parsrag").exists()), None)
repo_dir = local or Path("Persian-RAG-Academy")
if local is None and not repo_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{repo_dir}[st,openai]"], check=True)
print("parsrag installed from", repo_dir.resolve())

<div dir="rtl">

## ۲. بارگذاری پیکره و هر دو مجموعه پرسش

</div>

In [ ]:
import os
import platform
import subprocess
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np

import parsrag
from parsrag.eval import bootstrap_ci, evaluate_retrieval
from parsrag.eval.benchmark import index_documents, load_fixture_benchmark
from parsrag.normalize import normalize


def find_repo() -> Path:
    candidates = [os.environ.get("PARSRAG_REPO"), ".", "..", "Persian-RAG-Academy"]
    for c in candidates:
        if c and (Path(c) / "data" / "fixtures" / "qa.jsonl").exists():
            return Path(c).resolve()
    raise FileNotFoundError("Repository with data/fixtures not found: run the install cell first.")


REPO = find_repo()
FIXTURES = REPO / "data" / "fixtures"
OUT_DIR = Path("benchmark_results")
OUT_DIR.mkdir(exist_ok=True)

docs, qa_original = load_fixture_benchmark(FIXTURES, "qa.jsonl")
_, qa_paraphrase = load_fixture_benchmark(FIXTURES, "qa_paraphrase.jsonl")
QA_SETS = {"original": qa_original, "paraphrase": qa_paraphrase}

try:
    COMMIT = subprocess.run(
        ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    COMMIT = "unknown"

try:
    import torch

    DEVICE = f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"
except ImportError:
    DEVICE = "CPU"

print(f"parsrag {parsrag.__version__} ({COMMIT}), Python {platform.python_version()}, {DEVICE}")
print(f"{len(docs)} documents")
for name, qa in QA_SETS.items():
    answerable = sum(q.doc_id is not None for q in qa)
    print(f"{name}: {len(qa)} questions, {answerable} answerable (only these are scored)")

<div dir="rtl">

## ۳. تابع بنچمارک

برای هر مدل، پیکره **یک بار** قطعه‌بندی، نرمال‌سازی و برداری‌سازی می‌شود و هر دو مجموعه پرسش روی همان نمایه سنجیده می‌شوند.
این همان کاری است که `parsrag.eval.benchmark.benchmark_embedding` و `examples/benchmark_embeddings.py` می‌کنند (عمق بازیابی ۲۰، k = ۱، ۳، ۵).
برای MRR یک بازهٔ اطمینان ۹۵٪ با بوت‌استرپ (Bootstrap) هم حساب می‌کنیم تا ببینیم تفاوت دو مدل از نوسان کوچک‌بودن مجموعه بیشتر است یا نه.

</div>

In [ ]:
RESULTS: list[dict] = []


def run_benchmark(name: str, embedding, note: str = "") -> dict:
    """Index the corpus once with ``embedding`` and score both question sets (passage level)."""
    start = time.perf_counter()
    store = index_documents(docs, embedding)  # course default: recursive 400 + headings, normalised
    index_seconds = time.perf_counter() - start

    def retrieve(question: str, k: int):
        return store.search(normalize(question), k)

    row = {"model": name, "dim": embedding.dim, "note": note, "index_s": round(index_seconds, 1)}
    for set_name, qa in QA_SETS.items():
        start = time.perf_counter()
        report = evaluate_retrieval(qa, retrieve, (1, 3, 5), level="passage")
        rr = [1.0 / r.rank if r.rank else 0.0 for r in report.per_question]
        ci = bootstrap_ci(rr)
        row[set_name] = {
            "R@1": report.recall[1],
            "R@3": report.recall[3],
            "R@5": report.recall[5],
            "MRR": report.mrr,
            "MRR_ci": [ci.low, ci.high],
            "nDCG@3": report.ndcg[3],
            "n": report.n,
            "query_s": round(time.perf_counter() - start, 1),
        }
    RESULTS[:] = [r for r in RESULTS if r["model"] != name] + [row]
    o, p = row["original"], row["paraphrase"]
    print(f"{name:<50} MRR original {o['MRR']:.3f} | paraphrase {p['MRR']:.3f}")
    return row

<div dir="rtl">

## ۴. روش‌های آفلاین دوره (مرجع)

این سه روش هیچ چیزی دانلود نمی‌کنند و همان عددهای درس ۰۵ را باید بدهند (مثلاً `hashing` روی `qa.jsonl`: MRR = 0.841).
اگر عدد شما فرق دارد، احتمالاً نسخهٔ دیگری از مخزن یا داده‌ها را دارید.

</div>

In [ ]:
from parsrag.embeddings import HashingEmbedding, TfidfEmbedding

run_benchmark("hashing (offline)", HashingEmbedding())
run_benchmark("tfidf-word (offline)", TfidfEmbedding("word"))
run_benchmark("tfidf-char 2-4 (offline)", TfidfEmbedding("char", ngram_range=(2, 4)));

<div dir="rtl">

## ۵. مدل‌های چندزبانهٔ Hugging Face

نکته‌ها:

- خانوادهٔ **E5** (`intfloat/multilingual-e5-*`) با پیشوند آموزش دیده است: پرسش با `"query: "` و قطعه با `"passage: "`. بدون این پیشوندها نتیجه بدتر می‌شود؛ می‌توانید خودتان امتحان کنید.
- **bge-m3** و **paraphrase-multilingual-MiniLM** پیشوند لازم ندارند.
- `bge-m3` بزرگ‌تر است (حدود ۲٫۳ گیگابایت)؛ روی CPU دانلود و اجرا چند دقیقه طول می‌کشد.
- اولین اجرای هر مدل وزن‌ها را دانلود می‌کند؛ زمان `index_s` در جدول **بعد از** بارگذاری مدل اندازه گرفته می‌شود، پس زمان دانلود را شامل نمی‌شود.
- اگر یک مدل خطا داد (کمبود حافظه، قطع اینترنت)، بقیه ادامه می‌دهند.

مدل دیگری می‌خواهید؟ یک سطر به `MODELS` اضافه کنید: `(نام در Hugging Face, پیشوند پرسش, پیشوند قطعه)`.

</div>

In [ ]:
import gc

from parsrag.embeddings import SentenceTransformerEmbedding

MODELS = [
    # (Hugging Face model id, query prefix, passage prefix)
    ("intfloat/multilingual-e5-small", "query: ", "passage: "),
    ("intfloat/multilingual-e5-base", "query: ", "passage: "),
    ("BAAI/bge-m3", "", ""),
    ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "", ""),
    # ("intfloat/multilingual-e5-large", "query: ", "passage: "),
    # ("sentence-transformers/LaBSE", "", ""),
]

for model_id, query_prefix, doc_prefix in MODELS:
    embedding = None
    try:
        embedding = SentenceTransformerEmbedding(
            model_id, query_prefix=query_prefix, document_prefix=doc_prefix
        )
        note = f"prefixes {query_prefix.strip()!r}/{doc_prefix.strip()!r}" if query_prefix else ""
        run_benchmark(model_id, embedding, note)
    except Exception as exc:  # keep going with the other models
        print(f"{model_id}: FAILED — {type(exc).__name__}: {exc}")
    finally:
        del embedding
        gc.collect()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except ImportError:
            pass

<div dir="rtl">

## ۶. (اختیاری) هر سرویس سازگار با OpenAI

هر سرویس یا سرور محلی که `POST /embeddings` را به سبک OpenAI پیاده کرده باشد کار می‌کند (سرویس‌های ابری، vLLM، Ollama با `/v1` و...).
`USE_OPENAI_COMPATIBLE = True` کنید، آدرس و نام مدل را بنویسید و سلول را اجرا کنید.

- **کلید** با `getpass` پرسیده می‌شود: روی صفحه نمایش داده نمی‌شود، در دفترچه یا متغیر محیطی ذخیره نمی‌شود و بعد از اجرا پاک می‌شود. اگر سرویس کلید نمی‌خواهد، Enter بزنید.
- سرویس‌های پولی برای برداری‌سازی حدود ۱۰۰ قطعه و ۱۰۵ پرسش هزینهٔ کمی می‌گیرند، ولی **مسئولیت هزینه با شماست**؛ پیش از اجرا تعرفهٔ سرویس را ببینید.

</div>

In [ ]:
USE_OPENAI_COMPATIBLE = False
BASE_URL = "https://api.openai.com/v1"  # e.g. http://localhost:11434/v1 for Ollama
EMBEDDING_MODEL = "text-embedding-3-small"

if USE_OPENAI_COMPATIBLE:
    from getpass import getpass

    from parsrag.embeddings import OpenAICompatibleEmbedding

    api_key = getpass("API key (leave empty if not needed): ") or None
    try:
        run_benchmark(
            f"openai-compatible:{EMBEDDING_MODEL}",
            OpenAICompatibleEmbedding(EMBEDDING_MODEL, base_url=BASE_URL, api_key=api_key),
            note=BASE_URL,
        )
    finally:
        del api_key

<div dir="rtl">

## ۷. جدول نتیجه‌ها

جدول زیر با همان ستون‌های جدول «نتیجه‌های مدل‌های واقعی» در درس ۰۵ چاپ می‌شود، پس می‌توانید سطرهایش را مستقیم در آن جدول بگذارید.
نتیجه‌ها با جزئیات بیشتر (R@5، بازهٔ اطمینان MRR، زمان‌ها، نسخه‌ها) در `benchmark_results/results.json` هم ذخیره می‌شوند.

</div>

In [ ]:
import json

HEADER = (
    "| model | dim | orig R@1 | orig R@3 | orig MRR [95% CI] | orig nDCG@3 "
    "| para R@1 | para R@3 | para MRR [95% CI] | para nDCG@3 | hardware |"
)


def markdown_table(results: list[dict]) -> str:
    lines = [HEADER, "|" + "---|" * HEADER.count(" | ") + "---|"]
    for r in results:
        cells = [f"`{r['model']}`", str(r["dim"])]
        for set_name in ("original", "paraphrase"):
            m = r[set_name]
            lo, hi = m["MRR_ci"]
            mrr = f"{m['MRR']:.3f} [{lo:.2f}, {hi:.2f}]"
            cells += [f"{m['R@1']:.3f}", f"{m['R@3']:.3f}", mrr, f"{m['nDCG@3']:.3f}"]
        cells.append(DEVICE)
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


table = markdown_table(RESULTS)
meta = {
    "date": date.today().isoformat(),
    "parsrag": parsrag.__version__,
    "commit": COMMIT,
    "python": platform.python_version(),
    "device": DEVICE,
    "chunking": "RecursiveChunker(400, 50) + headings; parsrag.normalize on index and queries",
    "metrics": "passage level, depth 20, answerable questions only",
}
(OUT_DIR / "results.md").write_text(
    f"Measured {meta['date']}, parsrag {meta['parsrag']} ({meta['commit']}), {DEVICE}.\n\n"
    + table
    + "\n",
    encoding="utf-8",
)
(OUT_DIR / "results.json").write_text(
    json.dumps({"meta": meta, "results": RESULTS}, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(table)
print(f"\nSaved {OUT_DIR / 'results.md'} and {OUT_DIR / 'results.json'}")

In [ ]:
try:
    from IPython.display import Markdown, display

    display(Markdown(table))
except ImportError:
    pass

<div dir="rtl">

## ۸. نمودار

MRR هر مدل روی دو مجموعه پرسش، با بازهٔ اطمینان ۹۵٪. نمودار در `benchmark_results/embedding_benchmark.png` ذخیره می‌شود.

</div>

In [ ]:
import matplotlib

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")  # headless (CI or plain python)
import matplotlib.pyplot as plt

names = [r["model"].split("/")[-1] for r in RESULTS]
x = np.arange(len(RESULTS))
width = 0.38
fig, ax = plt.subplots(figsize=(max(7, 1.3 * len(RESULTS)), 4.5))
bars = ((-width / 2, "original", "#1e3a8a"), (width / 2, "paraphrase", "#60a5fa"))
for offset, set_name, color in bars:
    means = np.array([r[set_name]["MRR"] for r in RESULTS])
    lows = np.array([r[set_name]["MRR_ci"][0] for r in RESULTS])
    highs = np.array([r[set_name]["MRR_ci"][1] for r in RESULTS])
    ax.bar(
        x + offset,
        means,
        width,
        label=f"{set_name} questions",
        color=color,
        yerr=[means - lows, highs - means],
        capsize=3,
        error_kw={"elinewidth": 1},
    )
ax.set_xticks(x, names, rotation=25, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("passage MRR (95% bootstrap CI)")
ax.set_title(f"Persian fixtures: embedding models (parsrag {parsrag.__version__}, {DEVICE})")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "embedding_benchmark.png", dpi=150)
print("Saved", OUT_DIR / "embedding_benchmark.png")
plt.show()

<div dir="rtl">

## ۹. نتیجه‌ها را چطور بخوانیم؟

- **R@1**: سهم پرسش‌هایی که قطعهٔ اولِ بازیابی‌شده جملهٔ شاهد را دارد. برای RAG مهم است چون مدل زبانی بیشترین توجه را به قطعه‌های اول می‌کند.
- **R@3**: آیا قطعهٔ درست در سه قطعهٔ اول هست؟ اگر بعد از بازیابی بازرتبه‌بندی (درس ۰۸) دارید، این سنجه مهم‌تر از R@1 است.
- **MRR**: میانگین «یک تقسیم بر رتبهٔ اولین قطعهٔ درست». رتبهٔ ۱ = ۱، رتبهٔ ۲ = ۰٫۵، پیدا نشدن = ۰. یک عدد خلاصه برای مقایسه.
- **nDCG@3**: شبیه MRR ولی فقط سه قطعهٔ اول را می‌بیند و با لگاریتم رتبه وزن می‌دهد.
- **original در برابر paraphrase**: پرسش‌های `qa.jsonl` واژه‌های خود اسناد را تکرار می‌کنند و به نفع روش‌های واژگانی (TF-IDF، hashing) هستند. پرسش‌های بازنویسی‌شده به کاربر واقعی نزدیک‌ترند («کرایهٔ پیک» به‌جای «هزینهٔ ارسال»). **فاصلهٔ این دو ستون** نشان می‌دهد هر مدل چقدر واقعاً «معنا» را می‌فهمد.
- **بازهٔ اطمینان**: فقط ۵۹ و ۴۶ پرسش پاسخ‌دار داریم؛ یک پرسش حدود ۲ امتیاز درصد جابه‌جا می‌کند. اگر بازه‌های دو مدل خیلی هم‌پوشانی دارند، برتری یکی را ادعا نکنید (درس ۱۰: بوت‌استرپ جفتی).
- **dim و زمان** هم بخشی از تصمیم‌اند: بُعد بزرگ‌تر یعنی حافظهٔ بیشتر در انبارهٔ برداری (درس ۰۶) و مدل بزرگ‌تر یعنی تأخیر بیشتر برای هر پرسش.
- این پیکره کوچک و ساختگی است. مدلی که این‌جا برنده است لزوماً روی اسناد شما برنده نیست؛ با پرسش‌های خودتان دوباره بسنجید (درس ۱۳، `parsrag eval`).

</div>

<div dir="rtl">

## ۱۰. نتیجه را برگردانید (مشارکت)

نتیجهٔ اجرای شما می‌تواند به دیگران کمک کند. اگر خواستید:

1. فایل‌های `benchmark_results/results.md`، `results.json` و `embedding_benchmark.png` را دانلود کنید (سلول بعد در Colab این کار را می‌کند).
2. مخزن را Fork کنید و در `lessons/05-embeddings/README.fa.md` سطرهای مدل‌های واقعی را به جدول **«نتیجه‌های مدل‌های واقعی»** اضافه کنید — دقیقاً همان سطرهایی که دفترچه چاپ کرده، **بدون ویرایش دستی عددها**.
3. یک Pull Request باز کنید و در توضیحش بنویسید: تاریخ، commit مخزن، سخت‌افزار (ستون آخر) و محتوای `results.json`. اگر مدلی را با تنظیمات دیگری اجرا کرده‌اید (مثلاً بدون پیشوند E5)، این را هم بنویسید.

سطرهای روش‌های آفلاین را لازم نیست اضافه کنید؛ آن‌ها از قبل در درس هستند و CI بازتولیدشان را بررسی می‌کند.

</div>

In [ ]:
try:
    from google.colab import files  # only inside Colab

    for name in ("results.md", "results.json", "embedding_benchmark.png"):
        files.download(str(OUT_DIR / name))
except ImportError:
    print("Not running in Colab — the files are in", OUT_DIR.resolve())